# Week 3 · Your Week 2 work, reviewed

I read every cell you changed in Week 2 and every log entry. This notebook is short on purpose. You said you'd rather go through the notebooks in order than keep coming back to `00`, so **from this week on there's no notebook to return to.** Every notebook starts with its own 10-minute recap (section 0), and you go `00 → 01 → … → 08` once, in order. Section B shows where each thing you asked for is answered.

**How to use this notebook**
- Run cells one at a time with **Shift + Enter**.
- Before running an example, **guess what it prints**. The real result is written after `# →` in the comments, so you can check yourself.
- **Your turn** cells are for you. Open **Show one answer** only after a real attempt.
- If things get weird, click **Restart** at the top and run from the top again.

**Run this cell first.** It loads NumPy, the picture helpers (`figures.py` in this folder) and the exercise checker.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from figures import *

## A. Your Week 2 code and notes, reviewed

Most important first. Each point opens with a **Recap** box: what that exercise or lesson was and what you wrote, with a link (↗) back to it.

**A1. You were right: that table in 03 was confusing, and it was my mistake**

> ↩ **Recap** · [Week 2 · 03 §2, rule 2 (reductions) ↗](../week2/03_shapes_and_slicing.ipynb)
>
> Your comment on the table row `x.argmax(axis=1)` → `(3,)`: *"you made a mistake here. Dont make mistakes brother :("*

The table was about an array `x` with shape `(3, 4)`, and the sentence above it said so: "From shape `(3, 4)`". But when I added the picture of `sum(axis=0)` and `sum(axis=1)`, I put it **between** that sentence and the table. The picture uses `a` with shape `(2, 3)`, so the table looked like it described `a`. For `a`, `argmax(axis=1)` has shape `(2,)`, not `(3,)`:

In [1]:
import numpy as np
a = np.array([[1, 2, 3], [4, 5, 6]])     # the (2, 3) array from the picture
x = np.ones((3, 4))                      # the (3, 4) array the table meant
print(a.argmax(axis=1), a.argmax(axis=1).shape)   # → [2 2] (2,)
print(x.argmax(axis=1).shape)                     # → (3,)

[2 2] (2,)
(3,)


Your reading was right. The Study Pack page now says "a new array `x` with shape `(3, 4)`" right above the table. I left your notebook alone because your comment is in it.

**A2. Computing the gradient outside the loop is not a minor mistake**

> ↩ **Recap** · [Week 2 · 07 exercise 8 (logistic regression by gradient descent) ↗](../week2/07_exercises.ipynb)
>
> Repeat `steps` times: $w \leftarrow w - \eta \cdot \frac{1}{n} X^\top (\sigma(Xw) - y)$. **Your comment:** *"made a mistake where i had grad outside of the for iteration. minor mistake?"*

It's a big one, because it changes the algorithm. The gradient depends on $w$, and $w$ changes every step. If you compute it once, before the loop, every step uses the slope at the **starting point**, so you walk in a straight line forever and never stop at the bottom.

🖼 **Picture.** The same loss curve twice, starting at $w = 0$; the numbers are the step count. **Left:** the slope is measured again at every step, so the steps shrink as the curve flattens and stop at the bottom ($w = 2$). **Right:** the slope from $w = 0$ is reused, so every step is the same size and it walks straight past the bottom and up the other side.

In [ ]:
fig_frozen_gradient()

**Watching it happen**

One weight, data $x = (1, 2, 3)$, $y = (2, 4, 6)$, so the best $w$ is 2. The loss is $\frac{1}{3}\sum_i (w x_i - y_i)^2$ and its gradient is $\frac{2}{3} \sum_i x_i (w x_i - y_i) = \frac{28}{3}(w - 2)$. Start at $w = 0$ with $\eta = 0.05$. At $w = 0$ the gradient is $-18.67$, so the frozen version adds $0.05 \times 18.67 = 0.933$ every single step:

| Step | $w$ (correct) | loss (correct) | $w$ (frozen gradient) | loss (frozen) |
|---|---|---|---|---|
| 1 | 0.933 | 5.310 | 0.933 | 5.310 |
| 2 | 1.431 | 1.510 | 1.867 | 0.083 |
| 3 | 1.697 | 0.430 | 2.800 | 2.987 |
| 4 | 1.838 | 0.122 | 3.733 | 14.021 |
| 5 | 1.914 | 0.035 | 4.667 | 33.185 |
| 8 | 1.987 | 0.001 | 7.467 | 139.461 |

The correct version slows down as it approaches 2, because the slope there gets small. The frozen version passes 2 at step 3 and then the loss explodes. Your test only passed because you moved the line back inside the loop. **Rule: anything that depends on $w$ (predictions, error, gradient) must be computed inside the loop.**

**A3. The `np.clip` slip: trivial typo, important rule**

> ↩ **Recap** · [Week 2 · 07 exercise 7 (binary cross-entropy) ↗](../week2/07_exercises.ipynb)
>
> **Your comment:** *"The clipped result was thrown away. np.clip doesn't change p in place... this is kind of a trivial mistake so no need to review i believe. I leave the assessment to you."*

My assessment: the typo itself is trivial, but the question behind it isn't, and it comes back in pandas (this week) and PyTorch (Week 5). Every time you call something without assigning the result, ask: **does this change the object, or return a new one?**

| Call | Changes the original? | Returns | Where you met it |
|---|---|---|---|
| `np.clip(p, lo, hi)` | no | a new array | Week 2 · 07 ex. 7 |
| `sorted(lst)` | no | a new list | Week 0 |
| `lst.sort()`, `lst.append(x)` | yes | `None` | [Week 2 · 00 §A point 1 ↗](../week2/00_review_and_recap.ipynb): never store their result |
| `df.dropna()`, `df.sort_values(...)` | no | a new DataFrame | notebook 03 this week |
| `x.add_(1)` (PyTorch, trailing `_`) | yes | the same tensor | Week 5 |

In [ ]:
p = np.array([0.0, 0.5, 1.0])
np.clip(p, 0.01, 0.99)          # result thrown away
print(p)                        # → [0.  0.5 1. ]   unchanged
p = np.clip(p, 0.01, 0.99)      # keep the result
print(p)                        # → [0.01 0.5  0.99]

**A4. Your shape table is right. Use it as a habit**

> ↩ **Recap** · [Week 2 · 07, your note after exercise 12 ↗](../week2/07_exercises.ipynb)
>
> You listed your mistakes (`w = 0` instead of `np.zeros(d)`, `np.mean` collapsing the gradient, `y @ np.log(p)` summing instead of keeping one loss per sample, `np.mean(X_train)` giving one number, the unstored `np.clip`, the gradient outside the loop). You summed them up with one question: **"one per sample, one per feature, or one number?"**

That question is exactly right, and it covers every NumPy bug you had this week. Your last test cell shows the scaling one clearly:

Your test cell used `X = np.array([[1, 100], [3, 300]])`: two samples, two features on very different scales.

| Call | Averages over | Result | Shape | Kind |
|---|---|---|---|---|
| `np.mean(X)` | all 4 numbers | 101 | `()` | one number |
| `np.mean(X, axis=0)` | down each column | `[2, 200]` | `(2,)` | one per feature: what scaling needs |
| `np.mean(X, axis=1)` | across each row | `[50.5, 151.5]` | `(2,)` | one per sample |

| Scaled with | Result | Did it work? |
|---|---|---|
| one number: `(X - 101) / 121.66` | `[[-0.822, -0.008], [-0.806, 1.636]]` | no: feature 2 still dominates |
| per column: `(X - [2, 200]) / [1, 100]` | `[[-1, -1], [1, 1]]` | yes: each column has mean 0 and std 1 |

Notebook `02` turns your table into a workout: 10 short NumPy exercises where you predict the shape before you write the line.

**A5. Your three bugs in `gd_linear` were hiding each other**

> ↩ **Recap** · [Week 2 · 07 exercise 5 (gradient descent for linear regression) ↗](../week2/07_exercises.ipynb)
>
> Your first attempt: `w = 0`, then `w -= lr*2*np.mean(X.T@X*w - X.T@y)`. Your own diagnosis: (1) `w` should be a vector, (2) `np.mean` averaged the wrong thing, (3) `*` instead of `@`.

All three diagnoses are correct. One more thing to notice: bug 3 was **invisible** while bug 1 was there. With `w = 0` (a plain number), `X.T @ X * w` is just a matrix times a number, which is legal. Only after fixing bug 1 does `*` start doing the wrong thing. **Lesson: after fixing a bug, re-check the shapes of every line, not just the one you fixed.**

**A6. Ridge and the intercept: your note is right**

> ↩ **Recap** · [Week 2 · 07, your note after exercise 4 (ridge) ↗](../week2/07_exercises.ipynb)
>
> You wrote: the intercept $b$ should **not** be penalized, so set `I[0, 0] = 0` when the first column of `X` is the ones column; or center $X$ and $y$, fit without an intercept, and then $b = \bar y - \bar x^\top w$. "Same result."

All true, and here's how big the difference is. Data $x = (1, 2, 3, 4)$, $y = (52, 55, 53, 58)$, $\lambda = 2$:

| Version | $b$ | $w$ | Predictions |
|---|---|---|---|
| penalize $b$ too (full $I$) | 15.72 | 12.37 | 28.1, 40.5, 52.8, 65.2: nonsense |
| `I[0, 0] = 0` (your fix) | 51.64 | 1.14 | 52.8, 53.9, 55.1, 56.2 |
| scikit-learn `Ridge(alpha=2)` | 51.64 | 1.14 | the same: it centers, as you said |

Penalizing $b$ pulls it toward 0, but the data lives around 55, so the slope has to go wild to compensate. Week 2's exercise 4 had no intercept column, so the full $I$ was right there. In real use, always leave $b$ unpenalized.

In [ ]:
X = np.array([[1.0], [2.0], [3.0], [4.0]]); y = np.array([52.0, 55.0, 53.0, 58.0]); lam = 2.0
A = np.column_stack([np.ones(4), X])
I = np.eye(2); I[0, 0] = 0
print(np.linalg.solve(A.T @ A + lam * np.eye(2), A.T @ y).round(2))   # → [15.72 12.37]
print(np.linalg.solve(A.T @ A + lam * I, A.T @ y).round(2))           # → [51.64  1.14]
from sklearn.linear_model import Ridge
m = Ridge(alpha=lam).fit(X, y)
print(round(m.intercept_, 2), m.coef_.round(2))                       # → 51.64 [1.14]

**A7. kNN: your code works. Next you'll write it without the hint**

> ↩ **Recap** · [Week 2 · 07 exercise 11 (k-nearest neighbors) ↗](../week2/07_exercises.ipynb)
>
> You wrote two versions: a loop over test points (`np.linalg.norm(X_train - x, axis=1)`, then `np.argsort(dist)[:k]`, then `np.bincount(...).argmax()`), and the all-at-once version with `X_test[:, None, :] - X_train[None, :, :]`. Your log: *"i couldnt build kNN at all so maybe need some revision?"*

Both versions are correct, and every line has a shape comment, which is a great habit. You built them with the hint, though, and you want to be able to do it without. Notebook `02` rebuilds kNN from the smallest piece: one picture and one tiny exercise per line, so each line is something you derive rather than copy.

**A8. Your own notes: all correct, and several are better than mine**

| Where | Your note | Verdict |
|---|---|---|
| [01 §4 ↗](../week2/01_linear_algebra_review.ipynb) | Least squares as a projection, worked on $(0, 1), (1, 2), (2, 2)$, plus your picture | Correct. I checked your numbers: $a = \frac{7}{6}$, $b = 0.5$, residuals $(-\frac{1}{6}, \frac{1}{3}, -\frac{1}{6})$, at right angles to both columns. Your picture is the standard one in textbooks. |
| [02 §2 ↗](../week2/02_map_and_entropy_in_depth.ipynb) | "Narrow = confident", and "관측된 표본이 많아질수록 확실해지는 논리이다" | Correct. The row that got cut off should read: $\text{Beta}(50, 50)$, std 0.05, "$p$ is almost surely between 0.4 and 0.6". A $\text{Beta}(a, b)$ prior acts like $a - 1$ heads and $b - 1$ tails you've **already seen**, so $\text{Beta}(50, 50)$ is like 98 imaginary flips: it takes a lot of real data to move. |
| [02 §5–7 ↗](../week2/02_map_and_entropy_in_depth.ipynb) | Surprise $-\log p$; entropy = reality fills both slots; cross-entropy = reality sets the weights, the model sets the surprise; KL = the gap | Correct, and clearer than my version. Notebook `01` uses your surprise idea to explain the logistic loss, and `04` uses it for decision trees. |
| [04 §4 ↗](../week2/04_ml_intro_linear_regression.ipynb) | How many features is "too many" for the normal equation; the full gradient derivation | Correct. One more reason to avoid the normal equation: when features are almost copies of each other, $X^\top X$ is nearly singular and the solve becomes inaccurate. That's why scikit-learn's `LinearRegression` uses a least-squares solver instead. |
| [06 §3 ↗](../week2/06_pandas_and_sklearn.ipynb) | `stratify=y` keeps the class proportions the same in both splits | Correct. |

**What you did well**

- You debugged your own code and wrote down **why** each version was wrong. That's the best way to learn from a mistake.
- Your kNN has a shape comment on every line.
- You looked for extra explanations where mine weren't enough (projection, surprise, the threshold geometry) and checked them with numbers.
- Your log says exactly what didn't work for you (the 00 notebook, the logistic picture, PR-AUC, pandas practice), so this week is built from it.

## B. How Week 3 works

| You asked for | Where it is |
|---|---|
| "I'd rather finish the notebooks sequentially" | No hub notebook. Each notebook opens with its own recap (section 0). Go 00 → 08 once. |
| Logistic regression and its gradient step, with pictures | `01`, sections 1–8: every formula drawn on the same 9 points |
| The threshold trade-off and PR-AUC | `01`, sections 9–10 |
| kNN revision | `02`, sections 1–7 |
| "more trouble with numpy than the ML concept" | `02`, section 8: the shape workout |
| pandas methods with hands-on exercises | `03`: one method group per section, each followed by an exercise |
| The ML syllabus continues | `04` decision trees, `05` SVM, `06` ensembles, each ending with from-scratch exercises |
| (the exam is on paper) | `07`: exam-style problems solved by hand, and code written by hand |

**One thing missing from your log:** your Week 2 quiz scores. When you get a minute, add an entry like "Week 2 quiz: A 8/10, B 5/6, misses: A4 broadcasting, B2 ...". Misses by topic go into the recaps.

**The pictures.** Most sections now have a 🖼 **Picture** cell. Run it and the plot appears below. The drawing code is in `figures.py` in this folder if you're curious, but you don't need to read it.

---
Next: `01_logistic_regression_in_pictures.ipynb`.